# LeetCode 2249 — Count Lattice Points Inside a Circle bằng Python

Link đề: https://leetcode.com/problems/count-lattice-points-inside-a-circle/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Math, Geometry, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def countLatticePoints(self, circles: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Kiểm tra "điểm nằm trong (hoặc trên) đường tròn" bằng **số nguyên**: `(x − cx)² + (y − cy)² ≤ r²` — không cần căn bậc hai.
2. Đếm **hợp** của nhiều tập (điểm thuộc **ít nhất một** hình tròn) mà không đếm trùng: dùng `set`, mảng đánh dấu, hoặc **hợp đoạn** theo từng cột.
3. Dùng `math.isqrt` để tìm đoạn `y` hợp lệ trên mỗi cột `x` của một hình tròn — chính xác tuyệt đối, tránh lỗi làm tròn của `math.sqrt`.
4. Khai thác ràng buộc nhỏ (tọa độ ≤ 200) để chọn giữa "duyệt mọi điểm lưới" và "duyệt theo từng hình tròn".
5. Kiểm chứng các cách với nhau và với công thức đếm điểm của một hình tròn (bài toán đường tròn Gauss).

## 2. Đề bài & ví dụ

Cho mảng `circles`, `circles[i] = [x_i, y_i, r_i]` là tâm `(x_i, y_i)` và bán kính `r_i` của hình tròn thứ `i`. Trả về **số điểm nguyên** (lattice point — điểm có hai tọa độ nguyên) nằm **trong ít nhất một** hình tròn. Điểm **trên đường tròn** cũng tính là nằm trong.

**Ví dụ 1:** `circles = [[2,2,1]]` → `5`: các điểm `(1,2), (2,1), (2,2), (2,3), (3,2)`; `(1,1)` không tính vì `1² + 1² = 2 > 1`.

**Ví dụ 2:** `circles = [[2,2,2],[3,4,1]]` → `16`.

**Ràng buộc:** `1 ≤ circles.length ≤ 200`, `1 ≤ x_i, y_i ≤ 100`, `1 ≤ r_i ≤ min(x_i, y_i)` ⇒ mọi hình tròn nằm trong góc phần tư thứ nhất, mọi điểm cần xét có `0 ≤ x, y ≤ 200`.

In [1]:
from typing import List
from math import isqrt, sqrt
import random, time

examples = [([[2, 2, 1]], 5), ([[2, 2, 2], [3, 4, 1]], 16)]

def draw(circles, size=7):
    for y in range(size, -1, -1):
        row = ""
        for x in range(size + 1):
            inside = any((x - cx) ** 2 + (y - cy) ** 2 <= r * r for cx, cy, r in circles)
            row += "#" if inside else "."
        print(f"{y:>2} {row}")
    print("   " + "".join(str(x % 10) for x in range(size + 1)))

draw(examples[0][0]); print(); draw(examples[1][0])

 7 ........
 6 ........
 5 ........
 4 ........
 3 ..#.....
 2 .###....
 1 ..#.....
 0 ........
   01234567

 7 ........
 6 ........
 5 ...#....
 4 ..###...
 3 .###....
 2 #####...
 1 .###....
 0 ..#.....
   01234567


## 3. Kiến thức nền

### 3.1 Điều kiện nằm trong hình tròn bằng số nguyên

`(x, y)` nằm trong hoặc trên đường tròn tâm `(cx, cy)` bán kính `r` ⇔ khoảng cách ≤ `r` ⇔ `dx² + dy² ≤ r²` (bình phương hai vế không âm). Tất cả là số nguyên ⇒ không có sai số.

In [2]:
def inside(x, y, c):
    cx, cy, r = c
    return (x - cx) ** 2 + (y - cy) ** 2 <= r * r

print(inside(1, 2, [2, 2, 1]), inside(1, 1, [2, 2, 1]), inside(3, 2, [2, 2, 1]))

True False True


### 3.2 `math.isqrt` — căn bậc hai nguyên chính xác

`isqrt(k)` = số nguyên lớn nhất `t` với `t² ≤ k`. Dùng để tìm, trên cột `x`, nửa chiều cao `h = isqrt(r² − dx²)`: các `y` hợp lệ là `cy − h .. cy + h`. `int(sqrt(k))` có thể sai khi `k` là số chính phương lớn (do làm tròn float) — `isqrt` thì không.

In [3]:
for k in [0, 1, 3, 4, 24, 25, 26]:
    print(f"isqrt({k}) = {isqrt(k)}")
k = (10**8 + 1) ** 2 - 1
print("int(sqrt(k)) =", int(sqrt(k)), "| isqrt(k) =", isqrt(k), "| đúng là", 10**8)

isqrt(0) = 0
isqrt(1) = 1
isqrt(3) = 1
isqrt(4) = 2
isqrt(24) = 4
isqrt(25) = 5
isqrt(26) = 5
int(sqrt(k)) = 100000001 | isqrt(k) = 100000000 | đúng là 100000000


### 3.3 `set` các tuple để đếm hợp không trùng

Thêm `(x, y)` vào `set` nhiều lần chỉ giữ một bản ⇒ `len(set)` là số điểm phân biệt.

In [4]:
s = set()
s.add((1, 2)); s.add((1, 2)); s.add((2, 1))
print(s, len(s))

{(1, 2), (2, 1)} 2


### 3.4 Hợp các đoạn số nguyên trên một đường thẳng

Cho các đoạn `[l, r]` (số nguyên, bao gồm hai đầu), số điểm nguyên thuộc hợp: sắp xếp theo `l`, gộp đoạn giao nhau **hoặc kề nhau** (`l ≤ cur_r + 1`), cộng độ dài từng đoạn gộp.

In [5]:
def union_len(intervals):
    total = 0; cur_l = cur_r = None
    for l, r in sorted(intervals):
        if cur_r is None or l > cur_r + 1:
            if cur_r is not None: total += cur_r - cur_l + 1
            cur_l, cur_r = l, r
        else:
            cur_r = max(cur_r, r)
    if cur_r is not None: total += cur_r - cur_l + 1
    return total

print(union_len([(0, 2), (5, 6), (2, 4)]), union_len([(0, 1), (2, 3)]), union_len([(3, 3)]))

7 4 1


## 4. Ý tưởng nền tảng / trực giác

Có hai góc nhìn ngược nhau:

**Góc nhìn "từ điểm":** duyệt mọi điểm lưới trong vùng có thể (`0..200 × 0..200` ≈ 40 401 điểm), hỏi "có hình tròn nào chứa nó không?". Nếu dừng ngay khi tìm thấy hình tròn đầu tiên chứa nó thì không đếm trùng.

**Góc nhìn "từ hình tròn":** với mỗi hình tròn, liệt kê các điểm bên trong nó và gom vào một tập chung (hợp). Một hình tròn bán kính `r` chứa khoảng `πr²` điểm.

**Góc nhìn "theo cột":** cắt mặt phẳng thành các cột `x = const`. Trên mỗi cột, mỗi hình tròn cắt ra **một đoạn** `y` liên tục; hợp các đoạn ⇒ số điểm trên cột đó.

Tính tay ví dụ 1 theo cột (hình tròn `(2, 2, 1)`):

| x | dx | r² − dx² | h = isqrt | đoạn y | số điểm |
|---|---|---|---|---|---|
| 1 | −1 | 0 | 0 | [2, 2] | 1 |
| 2 | 0 | 1 | 1 | [1, 3] | 3 |
| 3 | 1 | 0 | 0 | [2, 2] | 1 |

Tổng `5`. ✔

In [6]:
cx, cy, r = 2, 2, 2
for x in range(cx - r, cx + r + 1):
    dx = x - cx; h = isqrt(r * r - dx * dx)
    print(f"x={x}: dx={dx:>2}, r²-dx²={r*r - dx*dx}, h={h}, đoạn y=[{cy - h}, {cy + h}] -> {2*h + 1} điểm")

x=0: dx=-2, r²-dx²=0, h=0, đoạn y=[2, 2] -> 1 điểm
x=1: dx=-1, r²-dx²=3, h=1, đoạn y=[1, 3] -> 3 điểm
x=2: dx= 0, r²-dx²=4, h=2, đoạn y=[0, 4] -> 5 điểm
x=3: dx= 1, r²-dx²=3, h=1, đoạn y=[1, 3] -> 3 điểm
x=4: dx= 2, r²-dx²=0, h=0, đoạn y=[2, 2] -> 1 điểm


## 5. Cách 1 — Brute Force / baseline: duyệt mọi điểm lưới × mọi hình tròn

**Ý tưởng:** khung bao `0..max(x+r) × 0..max(y+r)`; với mỗi điểm, kiểm tra từng hình tròn (dừng khi gặp hình tròn chứa nó).

In [7]:
def brute(circles):
    mx = max(x + r for x, _, r in circles); my = max(y + r for _, y, r in circles)
    cnt = 0
    for px in range(mx + 1):
        for py in range(my + 1):
            for cx, cy, r in circles:
                if (px - cx) ** 2 + (py - cy) ** 2 <= r * r:
                    cnt += 1
                    break
    return cnt

for c, exp in examples:
    got = brute(c)
    print(got, "OK" if got == exp else "SAI")

5 OK
16 OK


**Độ phức tạp:** O(X·Y·k) với `X, Y ≤ 201`, `k ≤ 200` ⇒ worst ≈ 8·10⁶ phép kiểm tra — chạy được nhưng chậm trong Python (vài giây) nếu không dừng sớm.

**Nút thắt:** phần lớn cặp (điểm, hình tròn) **ở rất xa nhau** — kiểm tra chúng là vô ích. Một hình tròn chỉ "liên quan" tới các điểm trong hình vuông bao của nó.

In [8]:
rng = random.Random(1)
worst = [[rng.randint(1, 5), rng.randint(1, 5), 1] for _ in range(200)] + [[100, 100, 100]]
t = time.perf_counter(); r1 = brute(worst); print(f"brute trên 201 hình tròn: {r1} điểm, {(time.perf_counter()-t)*1000:.0f} ms")

brute trên 201 hình tròn: 31462 điểm, 583 ms


(Trường hợp trên được dừng sớm nhờ hình tròn lớn đặt **cuối** danh sách — đổi thứ tự sẽ khác; xem Cách 5 về sắp xếp.)

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — chỉ duyệt trong hình vuông bao:** với hình tròn `(cx, cy, r)`, chỉ các điểm có `|x − cx| ≤ r`, `|y − cy| ≤ r` mới có thể nằm trong. Tổng công việc `Σ (2r + 1)²` thay vì `X·Y·k`.

**Insight 2 — không cần kiểm tra từng điểm trên một cột:** trên cột `x` (với `|dx| ≤ r`), tập `y` hợp lệ là **đoạn** `[cy − h, cy + h]` với `h = isqrt(r² − dx²)`. Vì vậy một hình tròn = `2r + 1` đoạn dọc.

**Insight 3 — đếm hợp theo cột:** gom mọi đoạn trên cùng cột `x` từ mọi hình tròn; số điểm trên cột = độ dài hợp các đoạn. Tổng O(Σ(2r+1) · log) — nhanh hơn liệt kê từng điểm khi `r` lớn.

**Quy tắc "nếu… thì…":**
- **Nếu** `dx² > r²` **thì** cột `x` không cắt hình tròn.
- **Nếu** hai đoạn trên cùng cột giao hoặc **kề** nhau (`l₂ ≤ r₁ + 1`) **thì** gộp.
- **Nếu** dùng `set` **thì** không cần lo trùng; **nếu** đếm theo hình tròn mà không dùng tập/hợp đoạn **thì** sẽ đếm trùng phần giao.

### 6.1 Ước lượng khối lượng công việc — chọn cách nào?

Số điểm nguyên trong hình tròn bán kính `r` xấp xỉ diện tích `πr²` (bài toán đường tròn Gauss: sai số cỡ `O(r^{2/3})`). Vậy:
- Cách "liệt kê điểm" tốn ≈ `Σ πr²` ≤ `200 · π · 100² ≈ 6,3·10⁶` — chấp nhận được.
- Cách "đoạn theo cột" tốn ≈ `Σ (2r + 1)` ≤ `200 · 201 ≈ 4·10⁴` đoạn — nhỏ hơn hai bậc.
- Cách "mọi điểm × mọi hình tròn" tốn `201² · 200 ≈ 8·10⁶` phép kiểm tra (không dừng sớm).

Cell dưới so số điểm thật với `πr²` để thấy ước lượng tốt cỡ nào:

In [9]:
import math
def gauss_count(r):
    return sum(2 * isqrt(r * r - x * x) + 1 for x in range(-r, r + 1))
for r in [1, 2, 5, 10, 50, 100]:
    g = gauss_count(r); a = math.pi * r * r
    print(f"r={r:>3}: số điểm = {g:>6}, πr² = {a:9.1f}, sai lệch = {g - a:7.1f}")

r=  1: số điểm =      5, πr² =       3.1, sai lệch =     1.9
r=  2: số điểm =     13, πr² =      12.6, sai lệch =     0.4
r=  5: số điểm =     81, πr² =      78.5, sai lệch =     2.5
r= 10: số điểm =    317, πr² =     314.2, sai lệch =     2.8
r= 50: số điểm =   7845, πr² =    7854.0, sai lệch =    -9.0
r=100: số điểm =  31417, πr² =   31415.9, sai lệch =     1.1


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Liệt kê điểm trong hình vuông bao, gom vào `set`

In [10]:
def set_union(circles):
    pts = set()
    for cx, cy, r in circles:
        rr = r * r
        for x in range(cx - r, cx + r + 1):
            for y in range(cy - r, cy + r + 1):
                if (x - cx) ** 2 + (y - cy) ** 2 <= rr:
                    pts.add((x, y))
    return len(pts)

for c, exp in examples:
    got = set_union(c)
    print(got, "OK" if got == exp else "SAI")

5 OK
16 OK


**Độ phức tạp cách 2:** O(Σ (2r+1)²) thời gian (≤ 200 · 201² ≈ 8·10⁶ worst, nhưng thường ít hơn nhiều), bộ nhớ O(số điểm hợp) ≤ 40 401.

### 7.2 Cách 3 — Mỗi cột một đoạn, thêm thẳng vào `set` (không thử từng `y`)

Kết hợp Insight 2 với `set`: với mỗi cột tính `h`, thêm cả đoạn `y` mà không cần kiểm tra điều kiện cho từng điểm.

In [11]:
def column_segments_set(circles):
    pts = set()
    for cx, cy, r in circles:
        for x in range(cx - r, cx + r + 1):
            h = isqrt(r * r - (x - cx) ** 2)
            pts.update((x, y) for y in range(cy - h, cy + h + 1))
    return len(pts)

for c, exp in examples:
    got = column_segments_set(c)
    print(got, "OK" if got == exp else "SAI")

5 OK
16 OK


Vẫn O(số điểm mỗi hình tròn) nhưng bỏ được ~21% số điểm (phần hình vuông ngoài hình tròn: `1 − π/4`).

### 7.3 Cách 4 — Hợp đoạn theo cột (không liệt kê điểm)

In [12]:
from collections import defaultdict
def column_union(circles):
    cols = defaultdict(list)
    for cx, cy, r in circles:
        for x in range(cx - r, cx + r + 1):
            h = isqrt(r * r - (x - cx) ** 2)
            cols[x].append((cy - h, cy + h))
    return sum(union_len(segs) for segs in cols.values())

for c, exp in examples:
    got = column_union(c)
    print(got, "OK" if got == exp else "SAI")

5 OK
16 OK


**Độ phức tạp cách 4:** số đoạn `S = Σ (2r + 1) ≤ 200 · 201`; sắp xếp mỗi cột ⇒ O(S log S). Không phụ thuộc diện tích — nếu bán kính lên tới 10⁶ thì cách này vẫn chạy, còn các cách liệt kê điểm thì không.

### 7.4 Cách 5 — Duyệt mọi điểm nhưng **sắp xếp hình tròn theo bán kính giảm dần** để dừng sớm

Hình tròn lớn chứa nhiều điểm ⇒ kiểm tra trước thì xác suất `break` sớm cao. Thêm cắt nhánh bằng hình vuông bao rẻ tiền trước phép nhân.

In [13]:
def brute_sorted(circles):
    cs = sorted({tuple(c) for c in circles}, key=lambda c: -c[2])      # khử trùng + lớn trước
    mx = max(x + r for x, _, r in cs); my = max(y + r for _, y, r in cs)
    cnt = 0
    for px in range(mx + 1):
        for py in range(my + 1):
            for cx, cy, r in cs:
                dx = px - cx; dy = py - cy
                if -r <= dx <= r and -r <= dy <= r and dx * dx + dy * dy <= r * r:
                    cnt += 1; break
    return cnt

for c, exp in examples:
    got = brute_sorted(c)
    print(got, "OK" if got == exp else "SAI")

5 OK
16 OK


### 7.5 Cách 6 — Mảng đánh dấu 2 chiều thay cho `set`

Tọa độ nằm trong `0..200` ⇒ dùng `bytearray` 201 × 201; đánh dấu theo đoạn cột (cắt lát gán) rất nhanh.

In [14]:
def mark_grid(circles):
    W = 201
    grid = [bytearray(W) for _ in range(W)]           # grid[x][y]
    for cx, cy, r in circles:
        for x in range(cx - r, cx + r + 1):
            h = isqrt(r * r - (x - cx) ** 2)
            col = grid[x]
            col[cy - h: cy + h + 1] = b"\x01" * (2 * h + 1)
    return sum(sum(col) for col in grid)

for c, exp in examples:
    got = mark_grid(c)
    print(got, "OK" if got == exp else "SAI")

5 OK
16 OK


Gán lát cắt `col[a:b] = b"\x01" * k` chạy trong C ⇒ cực nhanh. Tổng `sum(col)` đếm số ô bằng 1.

## 8. TRACE từng bước

### 8.1 Trace cách 4 (hợp đoạn theo cột) trên ví dụ 2

Hình tròn A `(2, 2, 2)`, hình tròn B `(3, 4, 1)`.

In [15]:
def trace_columns(circles):
    cols = defaultdict(list)
    for idx, (cx, cy, r) in enumerate(circles):
        for x in range(cx - r, cx + r + 1):
            h = isqrt(r * r - (x - cx) ** 2)
            cols[x].append((cy - h, cy + h, "AB"[idx] if idx < 2 else str(idx)))
    total = 0
    for x in sorted(cols):
        segs = sorted(cols[x]); u = union_len([(l, r) for l, r, _ in segs]); total += u
        print(f"x={x}: các đoạn {[(l, r, who) for l, r, who in segs]} -> hợp có {u} điểm, tổng={total}")
    return total

trace_columns(examples[1][0])

x=0: các đoạn [(2, 2, 'A')] -> hợp có 1 điểm, tổng=1
x=1: các đoạn [(1, 3, 'A')] -> hợp có 3 điểm, tổng=4
x=2: các đoạn [(0, 4, 'A'), (4, 4, 'B')] -> hợp có 5 điểm, tổng=9
x=3: các đoạn [(1, 3, 'A'), (3, 5, 'B')] -> hợp có 5 điểm, tổng=14
x=4: các đoạn [(2, 2, 'A'), (4, 4, 'B')] -> hợp có 2 điểm, tổng=16


16

Ở cột `x = 3`, đoạn của A là `[1, 3]` và của B là `[3, 5]` — giao nhau tại `y = 3` ⇒ hợp `[1, 5]` có 5 điểm (không phải 3 + 3 = 6). Ở cột `x = 4`, A cho `[2, 2]`, B cho `[4, 4]` — **không** kề (cách `y = 3`) ⇒ 2 điểm.

### 8.2 Trace cách 2 — đếm số điểm mỗi hình tròn đóng góp **mới** vào tập

In [16]:
def trace_set(circles):
    pts = set()
    for cx, cy, r in circles:
        before = len(pts)
        for x in range(cx - r, cx + r + 1):
            for y in range(cy - r, cy + r + 1):
                if (x - cx) ** 2 + (y - cy) ** 2 <= r * r: pts.add((x, y))
        print(f"hình tròn {(cx, cy, r)}: đóng góp {len(pts) - before} điểm mới (tổng {len(pts)})")
    return len(pts)

trace_set(examples[1][0])

hình tròn (2, 2, 2): đóng góp 13 điểm mới (tổng 13)
hình tròn (3, 4, 1): đóng góp 3 điểm mới (tổng 16)


16

Hình tròn B có 5 điểm nhưng chỉ đóng góp 3 điểm mới — 2 điểm `(3, 3)` và `(2, 4)` đã nằm trong A.

## 9. Code nộp LeetCode

In [17]:
from typing import List
from math import isqrt

class Solution:
    def countLatticePoints(self, circles: List[List[int]]) -> int:
        W = 201                                   # mọi tọa độ cần xét nằm trong [0, 200]
        grid = [bytearray(W) for _ in range(W)]   # grid[x][y] = 1 nếu đã được phủ
        for cx, cy, r in circles:
            rr = r * r
            for x in range(cx - r, cx + r + 1):
                h = isqrt(rr - (x - cx) * (x - cx))     # nửa chiều cao đoạn y trên cột x
                grid[x][cy - h: cy + h + 1] = b"\x01" * (2 * h + 1)
        return sum(sum(col) for col in grid)

In [18]:
sol = Solution()
for c, exp in examples:
    got = sol.countLatticePoints(c)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

5 | kỳ vọng 5 | OK
16 | kỳ vọng 16 | OK


## 10. Vì sao thuật toán đúng

**Đoạn trên một cột.** Cố định `x` với `|dx| ≤ r` (`dx = x − cx`). Điểm `(x, y)` thuộc hình tròn ⇔ `(y − cy)² ≤ r² − dx²` ⇔ `|y − cy| ≤ √(r² − dx²)` ⇔ (vì `y` nguyên) `|y − cy| ≤ ⌊√(r² − dx²)⌋ = isqrt(r² − dx²) = h`. Vậy tập `y` đúng là `[cy − h, cy + h]`. Với `|dx| > r` thì `r² − dx² < 0`, không có điểm nào — ta không duyệt các cột đó.

**Hợp không trùng.** Mảng đánh dấu (hay `set`, hay hợp đoạn) biểu diễn **tập** điểm: đánh dấu một ô nhiều lần vẫn chỉ là một ô ⇒ tổng số ô được đánh dấu = số điểm thuộc ít nhất một hình tròn.

**Đủ phạm vi.** Ràng buộc `r ≤ min(x, y)` và `x, y ≤ 100` ⇒ mọi điểm trong hình tròn có `0 ≤ x − r` và `x + r ≤ 200` (tương tự với `y`) ⇒ mảng 201 × 201 chứa đủ, không có chỉ số âm (Python sẽ hiểu chỉ số âm là "từ cuối" — một lỗi im lặng; ràng buộc đề đảm bảo không xảy ra). ∎

## 11. Phân tích Time / Space Complexity

`k ≤ 200` hình tròn, `R = max r ≤ 100`, `X, Y ≤ 201`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi điểm × mọi hình tròn | O(X·Y·k) | O(1) |
| 2. Hình vuông bao + set | O(Σ (2r+1)²) | O(X·Y) |
| 3. Đoạn cột + set | O(Σ số điểm trong hình tròn) | O(X·Y) |
| 4. Hợp đoạn theo cột | O(Σ(2r+1) · log) | O(Σ r) |
| 5. Mọi điểm, sắp xếp + dừng sớm | O(X·Y·k) worst, thực tế nhanh hơn | O(k) |
| 6. Mảng đánh dấu + gán lát cắt | O(Σ(2r+1) + X·Y) (gán lát cắt trong C) | O(X·Y) |

In [19]:
rng = random.Random(2249)
def rand_circles(k):
    out = []
    for _ in range(k):
        x, y = rng.randint(1, 100), rng.randint(1, 100); out.append([x, y, rng.randint(1, min(x, y))])
    return out
big = rand_circles(200)
res = {}
for name, f in [("brute", brute), ("set_union", set_union), ("column_segments_set", column_segments_set),
                ("column_union", column_union), ("brute_sorted", brute_sorted), ("mark_grid", mark_grid),
                ("Solution", Solution().countLatticePoints)]:
    t = time.perf_counter(); res[name] = f(big); dt = time.perf_counter() - t
    print(f"{name:<20} -> {res[name]}  {dt*1000:6.0f} ms")
assert len(set(res.values())) == 1

brute                -> 21872     176 ms


set_union            -> 21872      51 ms
column_segments_set  -> 21872      24 ms
column_union         -> 21872       2 ms
brute_sorted         -> 21872      61 ms
mark_grid            -> 21872       2 ms
Solution             -> 21872       2 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
ALL = {"brute": brute, "set_union": set_union, "column_segments_set": column_segments_set,
       "column_union": column_union, "brute_sorted": brute_sorted, "mark_grid": mark_grid,
       "Solution": Solution().countLatticePoints}
edge = examples + [
    ([[1, 1, 1]], 5),
    ([[2, 2, 1], [2, 2, 1]], 5),                  # hình tròn trùng nhau
    ([[3, 3, 3], [3, 3, 1]], 29),                 # hình tròn nhỏ nằm trong hình tròn lớn
    ([[2, 2, 1], [5, 5, 1]], 10),                 # tách rời
    ([[2, 2, 1], [4, 2, 1]], 9),                  # chạm nhau tại (3, 2)
    ([[100, 100, 100]], None),                    # so với công thức Gauss ở dưới
]
def gauss(r):                                     # số điểm nguyên trong hình tròn bán kính r
    return sum(2 * isqrt(r * r - x * x) + 1 for x in range(-r, r + 1))
for c, exp in edge:
    if exp is None: exp = gauss(c[0][2])
    for name, f in ALL.items():
        assert f([list(x) for x in c]) == exp, (name, c, f(c), exp)
print("Edge case OK; gauss(100) =", gauss(100), "≈ π·100² =", round(3.14159265 * 10**4))

Edge case OK; gauss(100) = 31417 ≈ π·100² = 31416


Các kỳ vọng tự dựng: `r = 3` có `gauss(3) = 29` điểm, nên hình tròn nhỏ bên trong không thêm gì; hai hình tròn bán kính 1 chạm nhau tại `(3, 2)` ⇒ `5 + 5 − 1 = 9`.

### 12.2 Đối chiếu ngẫu nhiên giữa mọi cách (seed cố định, đúng ràng buộc)

In [21]:
rng = random.Random(42)
for _ in range(300):
    k = rng.randint(1, 8)
    circ = []
    for _ in range(k):
        x, y = rng.randint(1, 12), rng.randint(1, 12)
        circ.append([x, y, rng.randint(1, min(x, y))])
    ref = brute(circ)
    for name, f in ALL.items():
        assert f([list(c) for c in circ]) == ref, (name, circ)
print("300 case ngẫu nhiên OK")

300 case ngẫu nhiên OK


### 12.3 Tính chất: bao hàm–loại trừ cho hai hình tròn; tịnh tiến không đổi đáp án

In [22]:
def pts_of(c):
    cx, cy, r = c
    return {(x, y) for x in range(cx - r, cx + r + 1) for y in range(cy - r, cy + r + 1) if (x - cx) ** 2 + (y - cy) ** 2 <= r * r}
rng = random.Random(7)
for _ in range(300):
    a = [rng.randint(5, 30), rng.randint(5, 30)]; a.append(rng.randint(1, 5))
    b = [rng.randint(5, 30), rng.randint(5, 30)]; b.append(rng.randint(1, 5))
    A, B = pts_of(a), pts_of(b)
    assert Solution().countLatticePoints([a, b]) == len(A) + len(B) - len(A & B)
    d = rng.randint(0, 50)
    assert Solution().countLatticePoints([[a[0] + d, a[1] + d, a[2]], [b[0] + d, b[1] + d, b[2]]]) == len(A | B)
print("tính chất OK")

tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `<` thay vì `<=` (bỏ điểm trên đường tròn)

In [23]:
def wrong_strict(circles):
    pts = set()
    for cx, cy, r in circles:
        for x in range(cx - r, cx + r + 1):
            for y in range(cy - r, cy + r + 1):
                if (x - cx) ** 2 + (y - cy) ** 2 < r * r: pts.add((x, y))     # SAI
    return len(pts)

print("bản sai :", wrong_strict([[2, 2, 1]]), "| bản đúng:", Solution().countLatticePoints([[2, 2, 1]]))

bản sai : 1 | bản đúng: 5


**Sửa:** đề nói điểm trên đường tròn cũng tính ⇒ `<=`.

### Lỗi 2 — Cộng số điểm của từng hình tròn (đếm trùng phần giao)

In [24]:
def wrong_sum_each(circles):
    return sum(len(pts_of(c)) for c in circles)

print("bản sai :", wrong_sum_each([[2, 2, 2], [3, 4, 1]]), "| bản đúng:", Solution().countLatticePoints([[2, 2, 2], [3, 4, 1]]))

bản sai : 18 | bản đúng: 16


**Sửa:** đếm **hợp** (set / mảng đánh dấu / hợp đoạn).

### Lỗi 3 — Dùng `int(math.sqrt(...))` cho `h`

Với bán kính ≤ 100, `int(sqrt(k))` thực ra luôn đúng (số nhỏ). Nhưng trong biến thể bán kính lớn thì sai — ta minh họa với một hình tròn bán kính `10⁸ + 1` chỉ trên **một cột** (không duyệt hết):

In [25]:
r = 10**8 + 1; dx = 1
k = r * r - dx * dx
print("int(sqrt):", int(sqrt(k)), "| isqrt:", isqrt(k), "| kiểm tra isqrt² <= k < (isqrt+1)²:", isqrt(k) ** 2 <= k < (isqrt(k) + 1) ** 2)
print("int(sqrt)² <= k ?", int(sqrt(k)) ** 2 <= k)

int(sqrt): 100000001 | isqrt: 100000000 | kiểm tra isqrt² <= k < (isqrt+1)²: True
int(sqrt)² <= k ? False


**Vì sao sai:** `sqrt` trả float 53 bit; với `k ≈ 10¹⁶` float làm tròn lên thành số nguyên kế tiếp ⇒ đoạn `y` bị nới thêm một điểm **nằm ngoài** hình tròn. **Sửa:** `math.isqrt`.

### Lỗi 4 — Khung duyệt chỉ tới 100 (quên `x + r` có thể tới 200)

In [26]:
def wrong_box(circles):
    cnt = 0
    for px in range(101):                                   # SAI: phải tới max(x + r)
        for py in range(101):
            if any((px - cx) ** 2 + (py - cy) ** 2 <= r * r for cx, cy, r in circles): cnt += 1
    return cnt

c = [[100, 100, 1]]
print("bản sai :", wrong_box(c), "| bản đúng:", Solution().countLatticePoints(c))

bản sai : 3 | bản đúng: 5


**Sửa:** khung `0..max(x_i + r_i)` × `0..max(y_i + r_i)`, hoặc cố định 201 × 201.

### Lỗi 5 — Lệch biên `range(cx − r, cx + r)` (thiếu cột cuối)

In [27]:
def wrong_range(circles):
    pts = set()
    for cx, cy, r in circles:
        for x in range(cx - r, cx + r):                       # SAI: thiếu x = cx + r
            h = isqrt(r * r - (x - cx) ** 2)
            pts.update((x, y) for y in range(cy - h, cy + h + 1))
    return len(pts)

print("bản sai :", wrong_range([[2, 2, 1]]), "| bản đúng:", Solution().countLatticePoints([[2, 2, 1]]))

bản sai : 4 | bản đúng: 5


**Sửa:** `range(cx − r, cx + r + 1)` (và tương tự cho `y`: `cy + h + 1`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Trong / trên hình tròn" với tọa độ nguyên ⇒ so **bình phương** khoảng cách với `r²`.
- "Ít nhất một" ⇒ **hợp** tập: `set`, mảng đánh dấu, hợp đoạn — tránh cộng thẳng.
- Ràng buộc tọa độ nhỏ (≤ 200) ⇒ dám duyệt cả lưới; ràng buộc lớn ⇒ chuyển sang **đoạn theo cột** (`isqrt`).
- `isqrt` thay cho `int(sqrt(...))`.
- Bài cùng họ: 1828 Queries on Number of Points Inside a Circle, 223 Rectangle Area (hợp hai hình), 850 Rectangle Area II (hợp nhiều hình — quét đường), 1401 Circle and Rectangle Overlapping.

## 15. Biến thể của bài

### 15.1 Biến thể A — Đếm điểm nằm trong **ít nhất k** hình tròn

Đếm số hình tròn phủ mỗi điểm bằng mảng đếm (không phải đánh dấu 0/1).

In [28]:
def at_least_k(circles, k):
    W = 201; cnt = [[0] * W for _ in range(W)]
    for cx, cy, r in circles:
        for x in range(cx - r, cx + r + 1):
            h = isqrt(r * r - (x - cx) ** 2)
            col = cnt[x]
            for y in range(cy - h, cy + h + 1): col[y] += 1
    return sum(1 for col in cnt for v in col if v >= k)

print(at_least_k([[2, 2, 2], [3, 4, 1]], 1), at_least_k([[2, 2, 2], [3, 4, 1]], 2))

16 2


### 15.2 Biến thể B — Bán kính và tọa độ rất lớn (tới 10⁶) nhưng ít hình tròn

Không thể đánh dấu từng điểm; dùng **hợp đoạn theo cột** (cách 4) — chỉ tốn O(Σ r) đoạn. Với `k = 3` hình tròn bán kính 10⁴ vẫn nhanh:

In [29]:
bigc = [[20000, 20000, 10000], [25000, 20000, 10000], [40000, 40000, 300]]
t = time.perf_counter(); v = column_union(bigc); dt = time.perf_counter() - t
print(f"{v} điểm, {dt*1000:.0f} ms (so với π·10⁸ ≈ 3,14·10⁸ điểm nếu liệt kê)")

413389982 điểm, 20 ms (so với π·10⁸ ≈ 3,14·10⁸ điểm nếu liệt kê)


### 15.3 Biến thể C — LC 1828: với mỗi **điểm** truy vấn, đếm số hình tròn chứa nó

Đảo vai: duyệt (điểm, hình tròn) — O(q·k), hoặc sắp xếp điểm theo `x` rồi chỉ xét điểm trong `[cx − r, cx + r]` bằng `bisect`.

In [30]:
from bisect import bisect_left, bisect_right
def count_points(points, queries):
    pts = sorted(map(tuple, points)); xs = [p[0] for p in pts]; out = []
    for cx, cy, r in queries:
        lo, hi = bisect_left(xs, cx - r), bisect_right(xs, cx + r)
        out.append(sum(1 for x, y in pts[lo:hi] if (x - cx) ** 2 + (y - cy) ** 2 <= r * r))
    return out

print(count_points([[1, 3], [3, 3], [5, 3], [2, 2]], [[2, 3, 1], [4, 3, 1], [1, 1, 2]]))   # [3, 2, 2]

[3, 2, 2]


### 15.4 Biến thể D — Đếm điểm trong **hình vành khăn** (trong hình tròn lớn, ngoài hình tròn nhỏ đồng tâm)

Trên mỗi cột, tập `y` là đoạn của hình tròn lớn **trừ** đoạn của hình tròn nhỏ ⇒ hiệu độ dài (vì đoạn nhỏ nằm trong đoạn lớn). Lưu ý "ngoài" ở đây nghĩa là **ngặt** ngoài: điểm trên đường tròn nhỏ không tính, nên đoạn bị trừ dùng `≤ r_nhỏ²`.

In [31]:
def annulus(cx, cy, R, r):
    total = 0
    for x in range(cx - R, cx + R + 1):
        dx2 = (x - cx) ** 2
        H = isqrt(R * R - dx2)
        h = isqrt(r * r - dx2) if dx2 <= r * r else -1      # -1: cột không cắt hình tròn nhỏ
        total += (2 * H + 1) - (2 * h + 1 if h >= 0 else 0)
    return total

def annulus_brute(cx, cy, R, r):
    return sum(1 for x in range(cx - R, cx + R + 1) for y in range(cy - R, cy + R + 1)
               if r * r < (x - cx) ** 2 + (y - cy) ** 2 <= R * R)

print(annulus(0, 0, 2, 1), annulus_brute(0, 0, 2, 1))
for R in range(1, 15):
    for r in range(0, R):
        assert annulus(0, 0, R, r) == annulus_brute(0, 0, R, r)
print("vành khăn: khớp brute force")

8 8
vành khăn: khớp brute force


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi điểm × hình tròn | đúng định nghĩa | O(X·Y·k) | O(1) | kiểm chứng |
| 2. Hình vuông bao + set | chỉ xét điểm gần | O(Σ(2r+1)²) | O(X·Y) | dễ viết |
| 3. Đoạn cột + set | bỏ góc hình vuông | O(Σ πr²) | O(X·Y) | gọn |
| 4. Hợp đoạn theo cột | không liệt kê điểm | O(Σ r · log) | O(Σ r) | **tọa độ lớn** |
| 5. Mọi điểm, sắp xếp | dừng sớm ở hình tròn lớn | O(X·Y·k) | O(k) | không dùng bộ nhớ phụ |
| 6. Mảng đánh dấu | gán lát cắt `bytearray` | O(Σ r + X·Y) | O(X·Y) | **mặc định**, nhanh nhất ở ràng buộc này |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `countLatticePoints([[2,2,2]])` và `countLatticePoints([[3,3,1],[3,3,2]])`?

**Bài 2 (tính tay):** Tính `h` trên mỗi cột cho hình tròn `(5, 5, 3)` và tổng số điểm.

**Bài 3 (điền TODO):** Viết `bt3_todo(r)` trả về số điểm nguyên trong hình tròn tâm gốc bán kính `r` (công thức Gauss) chỉ duyệt **một phần tư** hình tròn rồi suy ra.

**Bài 4 (tự viết lại):** Viết `bt4_todo(circles)` cài cách 4 (hợp đoạn theo cột) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(circles)` trả về số điểm nằm trong **đúng một** hình tròn.

In [32]:
def bt3_todo(r):
    # TODO: Gauss circle bằng một phần tư
    pass

def bt4_todo(circles):
    # TODO: hợp đoạn theo cột
    pass

def bt5_todo(circles):
    # TODO: số điểm trong ĐÚNG một hình tròn
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `gauss(2)`. Mức 2: cột: h = 0, 1, 2, 1, 0 ⇒ 1 + 3 + 5 + 3 + 1. Mức 3: `13`; ví dụ hai: hình tròn nhỏ nằm trong lớn ⇒ `13`.

**Bài 2.** Mức 1: `x = 2..8`. Mức 2: `h = 0, 2, 2, 3, 2, 2, 0`. Mức 3: tổng `1 + 5 + 5 + 7 + 5 + 5 + 1 = 29`.

**Bài 3.** Mức 1: đối xứng 4 góc phần tư. Mức 2: đếm điểm `x > 0, y ≥ 0` rồi nhân 4, cộng gốc. Mức 3: `1 + 4·Σ_{x=1..r} (isqrt(r² − x²) + 1)`.

**Bài 4.** Mức 1: `cols[x].append((cy − h, cy + h))`. Mức 2: sắp xếp, gộp khi `l ≤ cur_r + 1`. Mức 3: cộng độ dài hợp các cột.

**Bài 5.** Mức 1: mảng đếm như biến thể A. Mức 2: đếm ô có giá trị đúng bằng 1. Mức 3: `sum(1 for v in ... if v == 1)`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `13`, `13`. **Bài 2:** `29`.

In [33]:
assert Solution().countLatticePoints([[2, 2, 2]]) == 13
assert Solution().countLatticePoints([[3, 3, 1], [3, 3, 2]]) == 13
assert Solution().countLatticePoints([[5, 5, 3]]) == 29

def bt3(r):
    return 1 + 4 * sum(isqrt(r * r - x * x) + 1 for x in range(1, r + 1))

def bt4(circles):
    cols = defaultdict(list)
    for cx, cy, r in circles:
        for x in range(cx - r, cx + r + 1):
            h = isqrt(r * r - (x - cx) ** 2); cols[x].append((cy - h, cy + h))
    total = 0
    for segs in cols.values():
        segs.sort(); cl, cr = segs[0]
        for l, r in segs[1:]:
            if l <= cr + 1: cr = max(cr, r)
            else: total += cr - cl + 1; cl, cr = l, r
        total += cr - cl + 1
    return total

def bt5(circles):
    W = 201; cnt = [[0] * W for _ in range(W)]
    for cx, cy, r in circles:
        for x in range(cx - r, cx + r + 1):
            h = isqrt(r * r - (x - cx) ** 2)
            for y in range(cy - h, cy + h + 1): cnt[x][y] += 1
    return sum(1 for col in cnt for v in col if v == 1)

assert all(bt3(r) == gauss(r) for r in range(0, 150))
rng = random.Random(1949)
for _ in range(300):
    circ = []
    for _ in range(rng.randint(1, 6)):
        x, y = rng.randint(1, 15), rng.randint(1, 15); circ.append([x, y, rng.randint(1, min(x, y))])
    ref = brute(circ)
    assert bt4(circ) == ref
    sets = [pts_of(c) for c in circ]
    from collections import Counter
    cc = Counter(p for s_ in sets for p in s_)
    assert bt5(circ) == sum(1 for v in cc.values() if v == 1)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
grid = [bytearray(201) for _ in range(201)]
for cx, cy, r in circles:
    for x in range(cx - r, cx + r + 1):
        h = isqrt(r*r - (x - cx)**2)                 # đoạn y = [cy-h, cy+h]
        grid[x][cy - h: cy + h + 1] = b"\x01" * (2*h + 1)
return sum(sum(col) for col in grid)
```

**Checklist:**
- [ ] So `dx² + dy² <= r²` (có dấu bằng).
- [ ] Đếm **hợp**, không cộng từng hình tròn.
- [ ] `isqrt`, không `int(sqrt(...))`.
- [ ] Phạm vi tới `x + r`, `y + r` (≤ 200); `range(..., +1)` đủ hai đầu.
- [ ] Đã thử: hình tròn trùng nhau, lồng nhau, chạm nhau.